# PageIndex — Retrieval Walkthrough (When a Query Comes In)

This is the *runtime* counterpart to the indexing notebook. The tree from
the previous notebook is the artifact; this notebook shows what actually
happens, prompt-by-prompt, when a user asks a question.

## What changes vs vector RAG

In vector RAG, retrieval is a similarity computation: embed the query, do
top-k cosine over chunks, return text. PageIndex retrieval is a
**reasoning** step: the LLM is shown the document's table-of-contents
tree and asked which nodes likely contain the answer. There is no
embedding model and no vector database.

## Pipeline

```
question + tree
      │
      ▼
[1] strip text from tree     →   tree_without_text  (titles + summaries only)
      │
      ▼
[2] tree search prompt       →   LLM call           →  {"thinking": ..., "node_list": [...]}
      │                                                         │
      ▼                                                         │
[3] node_map lookup          ◀───────────────────────────────────┘
      │
      ▼
[4] gather text from selected nodes  →  context
      │
      ▼
[5] answer prompt            →   LLM call           →  final answer
```

There are **two LLM calls** per query: one for *which nodes*, one for
*what's the answer*. That's the entire baseline. Everything else is
variations on this theme:

- **Agentic retrieval** — instead of one tree-search call, expose
  `get_document_structure` and `get_page_content` as tools and let the
  agent iterate (PageIndex calls this *agentic vectorless RAG*).
- **Preference-aware retrieval** — same prompt, plus an extra
  "expert knowledge" block that biases which nodes get picked.
- **Cross-reference following** — re-run [2]–[3] when an answer node
  says "see Appendix G".

We'll cover the baseline in detail, then sketch the variations at the end.

---
## Step 0 — Setup

You need (a) the tree from the indexing notebook, (b) the original
`page_list` (so we can fetch page text), and (c) an async LLM caller.

If you ran the indexing notebook to completion you have:
- `pageindex_walkthrough_result.json` — the saved tree
- `page_list` in memory if you're in the same kernel

In [ ]:
import sys, os
REPO_ROOT = os.path.abspath("./PageIndex")  # adjust if needed
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# os.environ["OPENAI_API_KEY"] = "sk-..."


### 0.1 Load the tree

Either reload from the JSON we saved in the indexing notebook, or pass
in the live `final_output` dict from that kernel.

In [ ]:
import json

# Option A: reload from disk (what the indexing notebook produced)
with open("./pageindex_walkthrough_result.json", "r", encoding="utf-8") as f:
    final_output = json.load(f)

tree = final_output["structure"]
doc_description = final_output.get("doc_description")

print(f"Top-level sections: {len(tree)}")
print(f"Doc description:    {doc_description!r}")

# A peek:
for n in tree[:3]:
    print(f"  {n.get('node_id')}  {n.get('title')}  [pages {n.get('start_index')}–{n.get('end_index')}]")


### 0.2 Reload `page_list` from the source PDF

We need the actual page text to build the answer context. If `page_list`
is already in memory from the indexing notebook, skip this cell.

In [ ]:
from pageindex.utils import get_page_tokens

PDF_PATH = os.path.join(REPO_ROOT, "examples", "documents", "FRB_2023.pdf")  # same PDF as before
page_list = get_page_tokens(PDF_PATH, model="gpt-4o-2024-11-20")
print(f"Loaded {len(page_list)} pages.")


### 0.3 An async LLM caller

The cookbooks use OpenAI directly with `temperature=0` for
reproducibility. PageIndex's own `llm_acompletion` would also work — it
goes through LiteLLM so it's provider-agnostic. We'll use both in this
notebook so you can see them side by side.

In [ ]:
import openai

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")

async def call_llm(prompt: str, model: str = "gpt-4.1", temperature: float = 0) -> str:
    """Plain async OpenAI call. Same shape the cookbooks use."""
    client = openai.AsyncOpenAI(api_key=OPENAI_API_KEY)
    response = await client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
    )
    return response.choices[0].message.content.strip()

# PageIndex's own helper (works through LiteLLM, supports many providers):
from pageindex.utils import llm_acompletion, llm_completion

# Either is fine. The cookbook prefers call_llm; the indexing pipeline uses llm_acompletion.


---
## Step 1 — Strip the heavy `text` fields out of the tree

The tree from the indexing notebook may have `text` fields on every
node (the raw page content). Sending all of that to the search LLM
would (a) blow up the context window for large docs and (b) defeat the
purpose — we want the LLM to navigate by **titles + summaries**, not
re-read the document.

**Function:** `utils.remove_fields(tree, fields=[...])`

This is a simple recursive helper from `pageindex/utils.py`. We strip
`text` (and any other heavy fields you don't want the search LLM to see).

In [ ]:
from pageindex.utils import remove_fields
import copy

# Always operate on a copy — remove_fields mutates.
tree_without_text = remove_fields(copy.deepcopy(tree), fields=["text"])

# Show what the search LLM will actually see:
print(json.dumps(tree_without_text[:1], indent=2)[:1500])


**What's in `tree_without_text`?** For each node:
- `node_id` — the 4-digit ID assigned in indexing Step 9.1
- `title` — section title
- `summary` — the LLM-generated summary from indexing Step 9.3
- `start_index` / `end_index` — page span (kept so the LLM can reason about depth)
- `nodes` — children (recursively the same shape)

This is the "table of contents" the LLM gets to navigate. It's tiny —
typically < 5k tokens even for a 200-page report.

---
## Step 2 — Tree search: ask the LLM which nodes are relevant

This is the heart of PageIndex retrieval. One LLM call. The exact prompt
is from the official cookbook (`cookbook/pageindex_RAG_simple.ipynb`).

### 2.1 The tree search prompt

In [ ]:
TREE_SEARCH_PROMPT_TEMPLATE = """
You are given a question and a tree structure of a document.
Each node contains a node id, node title, and a corresponding summary.
Your task is to find all nodes that are likely to contain the answer to the question.

Question: {query}

Document tree structure:
{tree_without_text}

Please reply in the following JSON format:
{{
    "thinking": "<Your thinking process on which nodes are relevant to the question>",
    "node_list": ["node_id_1", "node_id_2", ..., "node_id_n"]
}}
Directly return the final JSON structure. Do not output anything else.
"""

print(TREE_SEARCH_PROMPT_TEMPLATE)


**A few things worth noticing about this prompt:**

- The LLM picks **multiple** nodes, not top-k by similarity. It gets to
  decide how many. For a narrow factual question, often just one node;
  for a "summarize the conclusions" type question, sometimes 3–5.
- The `thinking` field is non-optional. This is what gives PageIndex its
  *traceability* — every retrieval has a written rationale you can
  inspect, log, or show to the user.
- The tree is a JSON string inside a string. Big trees → keep summaries
  short (1–3 sentences each).

### 2.2 Run the tree search

In [ ]:
import asyncio

query = "What was the Federal Reserve's stance on financial stability in 2023?"

search_prompt = TREE_SEARCH_PROMPT_TEMPLATE.format(
    query=query,
    tree_without_text=json.dumps(tree_without_text, indent=2),
)

print(f"Prompt length (chars): {len(search_prompt):,}")
print(f"Prompt length (tokens, approx): {len(search_prompt) // 4:,}")
print()

tree_search_raw = asyncio.run(call_llm(search_prompt))
print("--- Raw LLM response ---")
print(tree_search_raw)


### 2.3 Parse the JSON response

`extract_json` is the same robust parser used in the indexing pipeline —
it handles ```json fences, trailing commas, and other LLM-isms.

In [ ]:
from pageindex.utils import extract_json

tree_search_result = extract_json(tree_search_raw)

print("--- thinking ---")
print(tree_search_result["thinking"])
print()
print("--- node_list ---")
print(tree_search_result["node_list"])


---
## Step 3 — Look up the selected nodes

We have `node_list = ["0007", "0019", ...]`. We need the actual node
objects (with `text`, `start_index`, `end_index`, etc.) so we can build
the answer context.

**Function:** `utils.create_node_mapping(tree)` — flattens the tree
into a `{node_id: node}` dict.

In [ ]:
from pageindex.utils import create_node_mapping

node_map = create_node_mapping(tree)  # use the ORIGINAL tree (with text), not tree_without_text

print(f"Total nodes in map: {len(node_map)}")
print()
print("--- Retrieved nodes ---")
for node_id in tree_search_result["node_list"]:
    if node_id not in node_map:
        print(f"  {node_id}  (NOT FOUND — LLM hallucinated this id)")
        continue
    n = node_map[node_id]
    s, e = n.get("start_index"), n.get("end_index")
    print(f"  {node_id}  pages {s}–{e}  {n.get('title')}")


**Defensive check.** Occasionally the LLM returns a node_id that
doesn't exist in the tree (a hallucinated ID, or a typo). Production
code should filter `node_list` against `node_map.keys()` before
proceeding. The cookbook skips this for brevity, but you should not.

---
## Step 4 — Build the answer context

Concatenate the text of the selected nodes. Two ways to do this:

**Option A — use the `text` field on the node** (only works if the
indexing notebook ran with `if_add_node_text="yes"`):

```python
relevant_content = "\n\n".join(node_map[nid]["text"] for nid in node_list)
```

**Option B — slice `page_list` by `start_index`/`end_index`** (always
works, doesn't require the heavy text to be stored on the tree):

In [ ]:
def gather_context_from_pages(node_ids, node_map, page_list):
    """Slice page_list by each node's [start_index, end_index] range."""
    seen_pages = set()
    pieces = []
    for nid in node_ids:
        if nid not in node_map:
            continue
        n = node_map[nid]
        s, e = n["start_index"], n["end_index"]
        for page_num in range(s, e + 1):
            if page_num in seen_pages:
                continue   # de-dupe across overlapping nodes
            seen_pages.add(page_num)
            pieces.append(f"[Page {page_num}]\n{page_list[page_num - 1][0]}")
    return "\n\n".join(pieces)

relevant_content = gather_context_from_pages(
    tree_search_result["node_list"], node_map, page_list
)

print(f"Context length (chars):  {len(relevant_content):,}")
print(f"Pages included:          {sum(1 for line in relevant_content.split(chr(10)) if line.startswith('[Page'))}")
print()
print("--- First 500 chars ---")
print(relevant_content[:500])


**Why this matters.** This is the moment where PageIndex differs
most from vector RAG. In vector RAG, you retrieve `top_k` chunks of
≤512 tokens each, hope the answer is in one of them, and pay the
fragmentation cost (the answer often spans chunk boundaries).

In PageIndex, you retrieve **complete sections at the granularity the
document itself uses** — a 3-page sub-section if that's what the LLM
picked, a single page if that's enough. No artificial boundaries.

---
## Step 5 — Generate the answer

Second (and final) LLM call. The cookbook uses a deliberately bare
prompt — the heavy lifting was done in Step 2.

### 5.1 The answer prompt

In [ ]:
ANSWER_PROMPT_TEMPLATE = """
Answer the question based on the context:

Question: {query}
Context: {relevant_content}

Provide a clear, concise answer based only on the context provided.
"""

print(ANSWER_PROMPT_TEMPLATE)


### 5.2 Call the LLM

In [ ]:
answer_prompt = ANSWER_PROMPT_TEMPLATE.format(
    query=query,
    relevant_content=relevant_content,
)

answer = asyncio.run(call_llm(answer_prompt))

print("=" * 60)
print(f"Q: {query}")
print("=" * 60)
print()
print(answer)
print()
print("=" * 60)
print("Sources:")
for nid in tree_search_result["node_list"]:
    if nid in node_map:
        n = node_map[nid]
        print(f"  - {n['title']}  (pages {n['start_index']}–{n['end_index']}, node {nid})")


**Citations come for free.** Because the LLM picked specific
node IDs in Step 2, you always know exactly which sections grounded the
answer. Vector RAG can give you "the chunks that scored highest" but
not "the section titled X starting at page Y" — you'd have to re-derive
that from chunk metadata.

---
## Putting it together — `ask(question, tree, page_list)`

The whole retrieval flow in one function:

In [ ]:
async def ask(question: str, tree: list, page_list: list, model: str = "gpt-4.1"):
    # 1. strip text from tree for the search prompt
    tree_without_text = remove_fields(copy.deepcopy(tree), fields=["text"])
    
    # 2. tree search
    search_prompt = TREE_SEARCH_PROMPT_TEMPLATE.format(
        query=question,
        tree_without_text=json.dumps(tree_without_text, indent=2),
    )
    search_raw = await call_llm(search_prompt, model=model)
    search_result = extract_json(search_raw)

    # 3. node map lookup (with hallucination filter)
    node_map = create_node_mapping(tree)
    valid_node_ids = [nid for nid in search_result["node_list"] if nid in node_map]

    # 4. build context
    context = gather_context_from_pages(valid_node_ids, node_map, page_list)

    # 5. answer
    answer_prompt = ANSWER_PROMPT_TEMPLATE.format(query=question, relevant_content=context)
    answer = await call_llm(answer_prompt, model=model)

    return {
        "question": question,
        "answer": answer,
        "thinking": search_result["thinking"],
        "node_ids": valid_node_ids,
        "sources": [
            {"node_id": nid, "title": node_map[nid]["title"],
             "pages": (node_map[nid]["start_index"], node_map[nid]["end_index"])}
            for nid in valid_node_ids
        ],
    }

# Try a few questions:
import asyncio
result = asyncio.run(ask(
    "What does the Fed say about monitoring financial vulnerabilities?",
    tree, page_list,
))
print(result["answer"])
print()
print("Sources:")
for s in result["sources"]:
    print(f"  - {s['title']} (pages {s['pages'][0]}–{s['pages'][1]})")


---
## Variation 1 — Tree search with expert preferences

If you have domain knowledge about *where* certain types of answers
typically live in your documents (e.g. "EBITDA questions → look at
Item 7 of a 10-K"), you can inject that into the search prompt without
fine-tuning anything.

Source: `docs.pageindex.ai/tutorials/tree-search/llm`

In [ ]:
TREE_SEARCH_WITH_PREFERENCE_PROMPT = """
You are given a question and a tree structure of a document.
You need to find all nodes that are likely to contain the answer.

Query: {query}
Document tree structure: {tree_without_text}
Expert Knowledge of relevant sections: {preference}

Reply in the following JSON format:
{{
    "thinking": <reasoning about which nodes are relevant>,
    "node_list": [node_id1, node_id2, ...]
}}
"""

# Example preference for a finance domain:
preference = (
    "If the query mentions EBITDA adjustments, prioritize Item 7 (MD&A) "
    "and footnotes in Item 8 (Financial Statements) in 10-K reports. "
    "If the query mentions financial stability or systemic risk, "
    "prioritize sections with 'Stability' or 'Vulnerabilities' in the title."
)

print(TREE_SEARCH_WITH_PREFERENCE_PROMPT.format(
    query="<your query>",
    tree_without_text="<tree>",
    preference=preference,
))


**When this is useful.** Compliance reviewers, financial
analysts, and other domain experts already have heuristics about where
to look. PageIndex lets you encode those heuristics as text and inject
them per-query — no embedding fine-tuning, no separate retrieval model.

---
## Variation 2 — Agentic retrieval

Instead of one tree-search call, expose the document as **tools** and
let an agent loop. The agent decides when to fetch the structure, which
pages to read, and when it has enough to answer. This is what
`examples/agentic_vectorless_rag_demo.py` in the repo demonstrates.

The three tools, all defined in `pageindex/retrieve.py`:

| Tool | What it returns |
|---|---|
| `get_document(doc_id)` | metadata: status, page count, name, description |
| `get_document_structure(doc_id)` | the tree without text fields (cheap to call repeatedly) |
| `get_page_content(doc_id, pages)` | text for specific pages, e.g. `"5-7"` or `"3,8"` |

These wrap your indexed data inside `PageIndexClient`, which persists
documents to a workspace folder.

### The agent system prompt

This is the exact prompt from `agentic_vectorless_rag_demo.py:38-46`:

In [ ]:
AGENT_SYSTEM_PROMPT = """
You are PageIndex, a document QA assistant.

TOOL USE:
- Call get_document() first to confirm status and page/line count.
- Call get_document_structure() to identify relevant page ranges.
- Call get_page_content(pages="5-7") with tight ranges; never fetch the whole document.
- Before each tool call, output one short sentence explaining the reason.

Answer based only on tool output. Be concise.
"""

print(AGENT_SYSTEM_PROMPT)


### Sketch of the loop (paraphrased from the official example)

```python
from pageindex import PageIndexClient
from agents import Agent, Runner, function_tool

client = PageIndexClient(workspace="./workspace")
doc_id = client.index(PDF_PATH)

@function_tool
def get_document() -> str:
    return client.get_document(doc_id)

@function_tool
def get_document_structure() -> str:
    return client.get_document_structure(doc_id)

@function_tool
def get_page_content(pages: str) -> str:
    return client.get_page_content(doc_id, pages)

agent = Agent(
    name="PageIndex",
    instructions=AGENT_SYSTEM_PROMPT,
    tools=[get_document, get_document_structure, get_page_content],
    model="gpt-4.1",
)

result = await Runner.run_streamed(agent, "Explain attention residuals.")
```

**When the agent loop wins over the manual two-step.** Multi-hop
questions ("What was the Q4 revenue, and how does it compare to the
forecast in the previous filing?") and questions that require following
in-document references ("see Appendix G"). The agent can call
`get_document_structure` again with new context after reading a section
and find the cross-referenced page.

**When the manual two-step is enough.** Single-hop factual questions
over a single document. You skip the agent overhead, you get
deterministic latency (exactly two LLM calls), and the prompts are
trivially auditable.

---
## Variation 3 — Vision-based answer generation

If your document has charts, tables, or figures that don't OCR cleanly,
do tree search exactly as in Step 2 (it only needs the titles and
summaries — those *are* OCR'd reliably), then for the **answer** call,
pass the actual page **images** to a VLM instead of the page text.

Source: `cookbook/vision_RAG_pageindex.ipynb`. The flow is identical
through Step 3, then Step 4/5 become:

In [ ]:
# Step 4 (vision variant): collect images of the selected pages
def get_page_images_for_nodes(node_ids, node_map, page_images):
    image_paths = []
    seen_pages = set()
    for node_id in node_ids:
        n = node_map[node_id]
        for page_num in range(n["start_index"], n["end_index"] + 1):
            if page_num not in seen_pages:
                image_paths.append(page_images[page_num])
                seen_pages.add(page_num)
    return image_paths

# (page_images is built once after indexing, by rasterizing the PDF with PyMuPDF)

# Step 5 (vision variant): answer prompt is shorter; the images carry the context
VISION_ANSWER_PROMPT_TEMPLATE = """
Answer the question based on the images of the document pages as context.

Question: {query}

Provide a clear, concise answer based only on the context provided.
"""

# Then call your VLM (e.g. GPT-4.1 multimodal) with prompt + image_paths
print(VISION_ANSWER_PROMPT_TEMPLATE)


**Why this works.** PageIndex's tree-search step only needs
*titles and summaries*, which OCR captures well. The hard part of the
document — figures, complicated tables — only enters the picture in the
final answer step, where the VLM can look at them directly. So the
"do we still need OCR?" question becomes: only barely, only for the
title-level structure of the document.

---
## All three patterns at a glance

| Pattern | LLM calls per query | Best for | Notebook reference |
|---|---|---|---|
| Manual two-step      | 2 (search + answer)            | Single-hop factual Qs, audit-friendly traces       | `cookbook/pageindex_RAG_simple.ipynb` |
| Agentic              | 3–10 (tools loop)              | Multi-hop, cross-references, mixed data sources    | `examples/agentic_vectorless_rag_demo.py` |
| Vision answer        | 2 (search + VLM answer)        | Documents with charts/figures/complex tables       | `cookbook/vision_RAG_pageindex.ipynb` |

All three share **the same indexing pipeline** (notebook 1) and **the
same tree search prompt** (Step 2 in this notebook). Only the answer
step changes.

---
## Function index — retrieval side

| Used in step | Function | File |
|---|---|---|
| 1 | `remove_fields(tree, fields=...)`            | `pageindex/utils.py` |
| 2 | `extract_json(raw)`                          | `pageindex/utils.py:94-125` |
| 2 | `llm_acompletion` / `llm_completion`         | `pageindex/utils.py:31-77` |
| 3 | `create_node_mapping(tree, ...)`             | `pageindex/utils.py` |
| Var 2 | `PageIndexClient.index`                  | `pageindex/client.py` |
| Var 2 | `PageIndexClient.get_document`           | `pageindex/client.py` |
| Var 2 | `PageIndexClient.get_document_structure` | `pageindex/client.py` |
| Var 2 | `PageIndexClient.get_page_content`       | `pageindex/client.py` (wraps `pageindex/retrieve.py`) |

## Prompt index — retrieval side

Every prompt the retrieval pipeline sends to an LLM, in one place:

1. **`TREE_SEARCH_PROMPT_TEMPLATE`** — the tree search prompt. The only
   prompt in baseline retrieval that does any reasoning. From
   `cookbook/pageindex_RAG_simple.ipynb`.
2. **`ANSWER_PROMPT_TEMPLATE`** — the answer prompt. Bare on purpose;
   the heavy lifting is in #1.
3. **`TREE_SEARCH_WITH_PREFERENCE_PROMPT`** — variant of #1 that injects
   domain knowledge. From `docs.pageindex.ai/tutorials/tree-search/llm`.
4. **`AGENT_SYSTEM_PROMPT`** — the system prompt for the agentic loop.
   From `examples/agentic_vectorless_rag_demo.py:38-46`.
5. **`VISION_ANSWER_PROMPT_TEMPLATE`** — variant of #2 for VLM-based
   answer generation. From `cookbook/vision_RAG_pageindex.ipynb`.